# Pursuer RL training

End-to-end notebook: env → PPO → evaluation → ONNX export.

Runs in Google Colab (GPU recommended) or local Jupyter. Paired files live at
https://github.com/unclestep/Rogue/tree/develop/rl. Trained model goes to
`rl/models/pursuer.onnx`; Go loads it via `ROGUE_PURSUER_MODEL_PATH`.

**Do not edit the observation layout without updating
`internal/domain/service/rl_observation.go` in lockstep.** The parity cell at
the bottom catches divergence, but only after the fact.


## 1. Setup

In [1]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Colab:' , IN_COLAB)


Colab: False


In [2]:
if IN_COLAB:
    %pip install --quiet 'gymnasium>=1.0' 'stable-baselines3>=2.4' torch onnx onnxscript onnxruntime tensorboard matplotlib


## 2. Hyperparameters

In [3]:
from pathlib import Path

CONFIG = {
    'topologies_dir': 'fixtures/topologies',  # swap for 'topologies' after make dump-topologies
    'total_timesteps': 5_000_000,
    'n_envs': 10,
    'max_episode_steps': 200,
    'seed': 42,
    'model_out': 'models/pursuer.onnx',
    'checkpoint_dir': 'checkpoints',
    'tensorboard_log': 'runs',
    # --- Architecture ---
    'net_arch': [256, 256, 128],  # was [128, 128]; 859-dim obs needs more capacity
    # --- Domain randomization ---
    'max_episode_steps_range': (120, 300),
    'distractor_prob': 0.3,
    'randomize_player_profile': True,
    # --- Early stopping ---
    # Evaluate on a held-out env every `eval_freq` env-steps and stop training
    # if mean reward hasn't improved by at least `es_min_delta` over
    # `es_patience` consecutive evaluations. Disables if `use_early_stopping`
    # is False (full `total_timesteps` run).
    'use_early_stopping': True,
    'eval_freq': 25_000,              # per-env step count
    'eval_episodes': 10,
    'es_patience': 6,
    'es_min_delta': 1.0,
    # --- Telemetry ---
    # SB3 verbose=1 floods the notebook; we record history via callback and
    # plot it in §5b. Keep 0 unless debugging the learner.
    'ppo_verbose': 0,
    'metrics_log_freq': 2_048,        # per-env step count
}
Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path('models').mkdir(parents=True, exist_ok=True)
CONFIG


{'topologies_dir': 'fixtures/topologies',
 'total_timesteps': 5000000,
 'n_envs': 10,
 'max_episode_steps': 200,
 'seed': 42,
 'model_out': 'models/pursuer.onnx',
 'checkpoint_dir': 'checkpoints',
 'tensorboard_log': 'runs',
 'net_arch': [256, 256, 128],
 'features_dim': 128,
 'max_episode_steps_range': (120, 300),
 'distractor_prob': 0.3,
 'randomize_player_profile': True,
 'use_early_stopping': True,
 'eval_freq': 25000,
 'eval_episodes': 10,
 'es_patience': 6,
 'es_min_delta': 1.0,
 'ppo_verbose': 0,
 'metrics_log_freq': 2048}

## 3. Environment modules

Sync the env/player modules into the notebook's working dir. In Colab this
pulls fresh copies from the repo; locally you can symlink or rely on the
current working directory being `rl/`.

In [4]:
import os, pathlib, shutil

if IN_COLAB:
    !git clone --depth=1 --filter=blob:none --sparse https://github.com/unclestep/Rogue.git _repo
    !cd _repo && git sparse-checkout set rl cmd/dump-topology
    for name in ('pursuer_env.py', 'scripted_player.py', '__init__.py'):
        src = pathlib.Path('_repo/rl') / name
        if src.exists():
            shutil.copy(src, name)
    shutil.copytree('_repo/rl/fixtures', 'fixtures', dirs_exist_ok=True)
else:
    # Local: assume we're running with cwd=rl/ (jupyter lab rl/pursuer_training.ipynb).
    pass

print('env files in cwd:', sorted(f for f in os.listdir('.') if f.endswith('.py')))


env files in cwd: ['__init__.py', 'features.py', 'pursuer_env.py', 'scripted_player.py']


In [5]:
# Import as local modules — both Colab-copied and local paths work.
import importlib, sys
sys.path.insert(0, '.')
import pursuer_env, scripted_player
importlib.reload(pursuer_env)
importlib.reload(scripted_player)
from pursuer_env import PursuerEnv, OBSERVATION_SIZE, ACTION_COUNT
from scripted_player import scripted_player_policy
print('observation_size =', OBSERVATION_SIZE, 'actions =', ACTION_COUNT)


observation_size = 859 actions = 5


## 4. Sanity check: single env

In [7]:
env = PursuerEnv(CONFIG['topologies_dir'], scripted_player_policy, max_episode_steps=50, seed=0)
obs, info = env.reset(seed=0)
print('obs', obs.shape, obs.dtype, 'range', obs.min(), obs.max())
print('loaded', info)
total = 0.0
for _ in range(30):
    obs, r, term, trunc, _ = env.step(env.action_space.sample())
    total += r
    if term or trunc: break
print('30-step random rollout reward =', round(total, 3))


obs (859,) float32 range 0.0 1.0
loaded {'topology': '0012.json', 'player_profile': 'default', 'has_distractor': False, 'max_steps': 50}
30-step random rollout reward = -0.07


## 5. PPO training

In [8]:
import time
from typing import Any

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CallbackList,
    CheckpointCallback,
    EvalCallback,
    StopTrainingOnNoModelImprovement,
)
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv, VecMonitor


def make_env(rank: int):
    def _thunk():
        return PursuerEnv(
            CONFIG['topologies_dir'],
            scripted_player_policy,
            max_episode_steps=CONFIG['max_episode_steps'],
            max_episode_steps_range=CONFIG.get('max_episode_steps_range'),
            distractor_prob=CONFIG.get('distractor_prob', 0.0),
            randomize_player_profile=CONFIG.get('randomize_player_profile', False),
            seed=CONFIG['seed'] + rank,
        )
    return _thunk


class MetricsHistoryCallback(BaseCallback):
    """Append SB3 logger stats to an in-memory dataframe every `log_freq` steps.

    SB3's `verbose=1` prints one table per rollout — for long runs that dumps
    thousands of lines into the notebook and bloats the .ipynb file. We read
    the same stats from `self.model.ep_info_buffer` and `self.model.logger`,
    keep them in Python lists, and plot them in the next cell.
    """

    def __init__(self, log_freq: int, verbose: int = 0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.history: dict[str, list[Any]] = {
            'timesteps': [], 'ep_rew_mean': [], 'ep_len_mean': [],
            'loss': [], 'policy_loss': [], 'value_loss': [], 'entropy_loss': [],
            'fps': [],
        }
        self._t0 = time.time()
        self._last_log = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last_log < self.log_freq:
            return True
        self._last_log = self.num_timesteps
        buf = self.model.ep_info_buffer
        if buf is None or len(buf) == 0:
            return True
        rewards = [ep['r'] for ep in buf]
        lengths = [ep['l'] for ep in buf]
        self.history['timesteps'].append(self.num_timesteps)
        self.history['ep_rew_mean'].append(float(np.mean(rewards)))
        self.history['ep_len_mean'].append(float(np.mean(lengths)))
        logger_vals = self.model.logger.name_to_value
        for k, dst in (('train/loss', 'loss'),
                       ('train/policy_gradient_loss', 'policy_loss'),
                       ('train/value_loss', 'value_loss'),
                       ('train/entropy_loss', 'entropy_loss')):
            self.history[dst].append(float(logger_vals.get(k, np.nan)))
        dt = max(time.time() - self._t0, 1e-6)
        self.history['fps'].append(self.num_timesteps / dt)
        return True


vec = SubprocVecEnv([make_env(i) for i in range(CONFIG['n_envs'])])
vec = VecMonitor(vec)

eval_vec = DummyVecEnv([make_env(1_000)])
eval_vec = VecMonitor(eval_vec)

model = PPO(
    'MlpPolicy',
    vec,
    device='cpu',
    learning_rate=3e-4,
    n_steps=1024,
    batch_size=512,
    n_epochs=5,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    policy_kwargs=dict(net_arch=CONFIG['net_arch']),
    tensorboard_log=CONFIG['tensorboard_log'],
    seed=CONFIG['seed'],
    verbose=CONFIG['ppo_verbose'],
)

ckpt_cb = CheckpointCallback(
    save_freq=max(10_000 // CONFIG['n_envs'], 1),
    save_path=CONFIG['checkpoint_dir'],
    name_prefix='pursuer',
)
metrics_cb = MetricsHistoryCallback(
    log_freq=CONFIG['metrics_log_freq'], verbose=0,
)
callbacks: list[BaseCallback] = [ckpt_cb, metrics_cb]

if CONFIG.get('use_early_stopping', False):
    stop_cb = StopTrainingOnNoModelImprovement(
        max_no_improvement_evals=CONFIG['es_patience'],
        min_evals=CONFIG['es_patience'],
        verbose=1,
    )
    eval_cb = EvalCallback(
        eval_vec,
        best_model_save_path=CONFIG['checkpoint_dir'],
        log_path=CONFIG['checkpoint_dir'],
        eval_freq=max(CONFIG['eval_freq'] // CONFIG['n_envs'], 1),
        n_eval_episodes=CONFIG['eval_episodes'],
        deterministic=True,
        render=False,
        callback_after_eval=stop_cb,
        verbose=1,
    )
    callbacks.append(eval_cb)

t_start = time.time()
model.learn(total_timesteps=CONFIG['total_timesteps'], callback=CallbackList(callbacks))
elapsed = time.time() - t_start
print(f'trained {model.num_timesteps:,} steps in {elapsed/60:.1f} min '
      f'({model.num_timesteps / elapsed:.0f} steps/s)')


Eval num_timesteps=25000, episode_reward=-37.01 +/- 101.43
Episode length: 201.60 +/- 48.94
New best mean reward!
Eval num_timesteps=50000, episode_reward=0.64 +/- 2.57
Episode length: 187.10 +/- 52.93
New best mean reward!
Eval num_timesteps=75000, episode_reward=-89.50 +/- 267.20
Episode length: 200.30 +/- 45.78
Eval num_timesteps=100000, episode_reward=-75.40 +/- 224.81
Episode length: 191.70 +/- 64.05
Eval num_timesteps=125000, episode_reward=-0.12 +/- 0.25
Episode length: 221.40 +/- 53.61
Eval num_timesteps=150000, episode_reward=-71.11 +/- 212.38
Episode length: 220.60 +/- 42.57
Eval num_timesteps=175000, episode_reward=0.48 +/- 2.07
Episode length: 201.30 +/- 60.36
Eval num_timesteps=200000, episode_reward=-45.66 +/- 140.71
Episode length: 226.00 +/- 62.54
Eval num_timesteps=225000, episode_reward=-185.74 +/- 439.18
Episode length: 220.10 +/- 49.80
Eval num_timesteps=250000, episode_reward=-0.84 +/- 5.08
Episode length: 190.80 +/- 74.17
Eval num_timesteps=275000, episode_reward=

## 5b. Learning curves

Plots from `MetricsHistoryCallback`. If `use_early_stopping` kicked in, the
curves stop before `total_timesteps`. Eval reward is read from
`checkpoints/evaluations.npz` written by `EvalCallback`.

In [9]:
import numpy as np

from pursuer_env import ACTION_VECTORS, ACTION_WAIT, chase_scent_map


class BaselinePolicy:
    """Dijkstra scent-chase with oracle knowledge of player_pos.

    Mirrors Go's `dijkstraCardinal.findOptimals` (sys_pathfinder.go:128) —
    picks any cardinal whose scent ≤ center, ties broken deterministically by
    Manhattan distance to the player (Go randomises; we want reproducibility).

    This is the skill floor: RL must beat it on stealth/ambush to justify the
    whole stack. It has *strictly better information* (no LOS, no decay).
    """

    def __init__(self, env):
        self._env = env

    def predict(self, obs, deterministic=True):
        env = self._env
        assert env.topology is not None
        scent = chase_scent_map(env.topology, env.player_pos)
        ox, oy = env.pursuer_pos
        center_val = int(scent[oy, ox])

        candidates = []
        min_val = center_val
        for action, (dx, dy) in ACTION_VECTORS.items():
            if action == ACTION_WAIT:
                continue
            nx, ny = ox + dx, oy + dy
            if not env.topology.is_walkable(nx, ny):
                continue
            v = int(scent[ny, nx])
            if v < min_val:
                min_val = v
                candidates = [(action, nx, ny)]
            elif v == min_val:
                candidates.append((action, nx, ny))

        if not candidates:
            return ACTION_WAIT, None
        px, py = env.player_pos
        action, _, _ = min(
            candidates,
            key=lambda c: (abs(c[1] - px) + abs(c[2] - py), c[0]),
        )
        return action, None


def rollout(policy_name, policy, eval_env, n_episodes=30, seed_base=1000):
    returns, lengths, in_cone_ratios = [], [], []
    catches, total_hits, ambush_hits = 0, 0, 0
    pursuer_deaths = 0
    pursuer_hp_end = []
    first_hit_turns = []
    for ep in range(n_episodes):
        obs, _ = eval_env.reset(seed=seed_base + ep)
        ep_ret, ep_steps, ep_in_cone = 0.0, 0, 0
        ep_hits, ep_ambush = 0, 0
        first_hit = None
        done = False
        info = {}
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            obs, r, term, trunc, info = eval_env.step(int(action))
            ep_ret += r
            ep_steps += 1
            if info.get('in_cone'):
                ep_in_cone += 1
            if info.get('pursuer_hit'):
                ep_hits += 1
                if first_hit is None:
                    first_hit = ep_steps
            if info.get('ambush_hit'):
                ep_ambush += 1
            done = term or trunc
        returns.append(ep_ret)
        lengths.append(ep_steps)
        in_cone_ratios.append(ep_in_cone / max(ep_steps, 1))
        total_hits += ep_hits
        ambush_hits += ep_ambush
        pursuer_hp_end.append(int(info.get('pursuer_hp', 0)))
        if info.get('pursuer_hp', 1) <= 0:
            pursuer_deaths += 1
        if info.get('caught'):
            catches += 1
        if first_hit is not None:
            first_hit_turns.append(first_hit)
    return {
        'policy': policy_name,
        'avg_return': float(np.mean(returns)),
        'std_return': float(np.std(returns)),
        'catch_rate': catches / n_episodes,
        'avg_in_cone': float(np.mean(in_cone_ratios)),
        'avg_length': float(np.mean(lengths)),
        'avg_first_hit_turn': float(np.mean(first_hit_turns)) if first_hit_turns else float('nan'),
        'total_hits': total_hits,
        'ambush_hits': ambush_hits,
        'ambush_ratio': ambush_hits / max(total_hits, 1),
        'death_rate': pursuer_deaths / n_episodes,
        'avg_hp_end': float(np.mean(pursuer_hp_end)),
    }


eval_env = PursuerEnv(
    CONFIG['topologies_dir'],
    scripted_player_policy,
    max_episode_steps=CONFIG['max_episode_steps'],
    seed=999,
)

results = [
    rollout('PPO', model, eval_env, n_episodes=30),
    rollout('Baseline (chase-scent)', BaselinePolicy(eval_env), eval_env, n_episodes=30),
]

header = (
    f"{'policy':<24} {'return':>14} {'catch':>7} {'in_cone':>9} "
    f"{'len':>6} {'1st_hit':>8} {'hits':>6} {'ambush':>8} "
    f"{'death':>7} {'hp_end':>8}"
)
print(header)
print('-' * len(header))
for r in results:
    first_hit = (
        f"{r['avg_first_hit_turn']:>8.1f}"
        if r['avg_first_hit_turn'] == r['avg_first_hit_turn']
        else f"{'nan':>8}"
    )
    print(
        f"{r['policy']:<24} "
        f"{r['avg_return']:>7.2f} ± {r['std_return']:>4.2f} "
        f"{r['catch_rate']:>7.1%} "
        f"{r['avg_in_cone']:>9.3f} "
        f"{r['avg_length']:>6.1f} "
        f"{first_hit} "
        f"{r['total_hits']:>6d} "
        f"{r['ambush_ratio']:>8.1%} "
        f"{r['death_rate']:>7.1%} "
        f"{r['avg_hp_end']:>8.1f}"
    )

# Interpretation cheat sheet:
#   catch_rate        — % of episodes where player died.
#   death_rate        — % where the pursuer itself got killed (counter-attacks).
#   avg_hp_end        — pursuer HP at episode end; low means the policy is
#                       melee-trading instead of hit-and-run.
#   in_cone / ambush  — stealth quality; lower in_cone and higher ambush_ratio
#                       than baseline is the signature of a good policy.


policy                           return   catch   in_cone    len  1st_hit   hits   ambush   death   hp_end
----------------------------------------------------------------------------------------------------------
PPO                        -0.51 ± 0.93    0.0%     0.001  200.0      nan      0     0.0%    0.0%     40.0
Baseline (chase-scent)     19.31 ± 20.83   33.3%     0.046   51.6     53.0     30    73.3%   66.7%     12.7


## 6. Evaluation

In [10]:
import numpy as np

from pursuer_env import ACTION_VECTORS, ACTION_WAIT, chase_scent_map


class BaselinePolicy:
    """
    Greedy chase against the player's *true* position — same algorithm as Go's
    `dijkstraCardinal.findOptimals` (sys_pathfinder.go:128): the center cell
    only acts as a baseline for `minScent`, and any cardinal neighbor whose
    scent is ≤ center gets considered; among the tied-minimum candidates, Go
    picks randomly. Here we tie-break deterministically by Manhattan distance
    to the player so eval is reproducible.

    This is the skill floor: the RL policy must outperform it on stealth/
    ambush metrics, otherwise RL adds no value over hand-coded ChaseBehavior.
    It has *strictly better information* than the Pursuer — reads player_pos
    directly, no LOS or memory decay. That's intentional.
    """

    def __init__(self, env):
        self._env = env

    def predict(self, obs, deterministic=True):
        env = self._env
        assert env.topology is not None
        scent = chase_scent_map(env.topology, env.player_pos)
        ox, oy = env.pursuer_pos
        center_val = int(scent[oy, ox])

        candidates = []
        min_val = center_val
        for action, (dx, dy) in ACTION_VECTORS.items():
            if action == ACTION_WAIT:
                continue
            nx, ny = ox + dx, oy + dy
            if not env.topology.is_walkable(nx, ny):
                continue
            v = int(scent[ny, nx])
            if v < min_val:
                min_val = v
                candidates = [(action, nx, ny)]
            elif v == min_val:
                candidates.append((action, nx, ny))

        if not candidates:
            return ACTION_WAIT, None
        # Deterministic tie-break: cardinal with smallest Manhattan distance
        # to the player (Go randomises; we want reproducible eval runs).
        px, py = env.player_pos
        action, _, _ = min(
            candidates,
            key=lambda c: (abs(c[1] - px) + abs(c[2] - py), c[0]),
        )
        return action, None


def rollout(policy_name, policy, eval_env, n_episodes=30, seed_base=1000):
    returns, lengths, in_cone_ratios = [], [], []
    catches, total_hits, ambush_hits = 0, 0, 0
    first_hit_turns = []
    for ep in range(n_episodes):
        obs, _ = eval_env.reset(seed=seed_base + ep)
        ep_ret, ep_steps, ep_in_cone = 0.0, 0, 0
        ep_hits, ep_ambush = 0, 0
        first_hit = None
        done = False
        info = {}
        while not done:
            action, _ = policy.predict(obs, deterministic=True)
            obs, r, term, trunc, info = eval_env.step(int(action))
            ep_ret += r
            ep_steps += 1
            if info.get('in_cone'):
                ep_in_cone += 1
            if info.get('pursuer_hit'):
                ep_hits += 1
                if first_hit is None:
                    first_hit = ep_steps
            if info.get('ambush_hit'):
                ep_ambush += 1
            done = term or trunc
        returns.append(ep_ret)
        lengths.append(ep_steps)
        in_cone_ratios.append(ep_in_cone / max(ep_steps, 1))
        total_hits += ep_hits
        ambush_hits += ep_ambush
        if info.get('caught'):
            catches += 1
        if first_hit is not None:
            first_hit_turns.append(first_hit)
    return {
        'policy': policy_name,
        'avg_return': float(np.mean(returns)),
        'std_return': float(np.std(returns)),
        'catch_rate': catches / n_episodes,
        'avg_in_cone': float(np.mean(in_cone_ratios)),
        'avg_length': float(np.mean(lengths)),
        'avg_first_hit_turn': float(np.mean(first_hit_turns)) if first_hit_turns else float('nan'),
        'total_hits': total_hits,
        'ambush_hits': ambush_hits,
        'ambush_ratio': ambush_hits / max(total_hits, 1),
    }


eval_env = PursuerEnv(
    CONFIG['topologies_dir'],
    scripted_player_policy,
    max_episode_steps=CONFIG['max_episode_steps'],
    seed=999,
)

results = [
    rollout('PPO', model, eval_env, n_episodes=30),
    rollout('Baseline (chase-scent)', BaselinePolicy(eval_env), eval_env, n_episodes=30),
]

header = (
    f"{'policy':<24} {'return':>14} {'catch':>7} {'in_cone':>9} "
    f"{'len':>6} {'1st_hit':>8} {'hits':>6} {'ambush':>8}"
)
print(header)
print('-' * len(header))
for r in results:
    first_hit = (
        f"{r['avg_first_hit_turn']:>8.1f}"
        if r['avg_first_hit_turn'] == r['avg_first_hit_turn']
        else f"{'nan':>8}"
    )
    print(
        f"{r['policy']:<24} "
        f"{r['avg_return']:>7.2f} ± {r['std_return']:>4.2f} "
        f"{r['catch_rate']:>7.1%} "
        f"{r['avg_in_cone']:>9.3f} "
        f"{r['avg_length']:>6.1f} "
        f"{first_hit} "
        f"{r['total_hits']:>6d} "
        f"{r['ambush_ratio']:>8.1%}"
    )

# The RL policy wins this comparison when it lowers `in_cone` and lifts
# `ambush` without losing too much on `catch`/`1st_hit`. If PPO trails the
# baseline on every axis after a real training run, reward shaping needs
# tuning (see PR 5).


policy                           return   catch   in_cone    len  1st_hit   hits   ambush
-----------------------------------------------------------------------------------------
PPO                        -0.51 ± 0.93    0.0%     0.001  200.0      nan      0     0.0%
Baseline (chase-scent)     19.31 ± 20.83   33.3%     0.046   51.6     53.0     30    73.3%


## 7. ONNX export

Export a torch module that maps flat observation → raw logits, matching
Go's `ONNXPolicy.Predict` (argmax done in Go). Input name `input`, output
name `logits`, opset 17.

In [12]:
import torch
import torch.nn as nn

class PursuerActor(nn.Module):
    """Wraps the SB3 policy so torch.onnx.export sees a clean forward(obs) -> logits."""

    def __init__(self, sb3_policy):
        super().__init__()
        self.policy = sb3_policy

    def forward(self, obs):
        # SB3 >= 2.0 layout. Bump the check here if you upgrade SB3.
        features = self.policy.extract_features(obs, self.policy.pi_features_extractor)
        latent_pi, _ = self.policy.mlp_extractor(features)
        return self.policy.action_net(latent_pi)

actor = PursuerActor(model.policy).eval()
dummy = torch.zeros(1, OBSERVATION_SIZE)
torch.onnx.export(
    actor,
    dummy,
    CONFIG['model_out'],
    input_names=['input'],
    output_names=['logits'],
    opset_version=17,
    dynamic_axes=None,  # fixed shape — Go sends [1, 859] exactly
)
print('exported', CONFIG['model_out'])


W0418 20:41:45.691000 2499 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0418 20:41:45.981000 2499 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::nms
W0418 20:41:45.981000 2499 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_align
W0418 20:41:45.982000 2499 torch/onnx/_internal/exporter/_registration.py:110] torchvision is not installed. Skipping torchvision::roi_pool


[torch.onnx] Obtain model graph for `PursuerActor([...]` with `torch.export.export(..., strict=False)`...


/opt/homebrew/Cellar/python@3.14/3.14.3_1/Frameworks/Python.framework/Versions/3.14/lib/python3.14/copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Obtain model graph for `PursuerActor([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...
[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
exported models/pursuer.onnx


## 8. Parity check — PyTorch vs ONNX

In [13]:
import onnxruntime as ort

sess = ort.InferenceSession(CONFIG['model_out'])
rng = np.random.default_rng(0)
max_diff = 0.0
for _ in range(1000):
    sample = rng.uniform(-1, 1, size=(1, OBSERVATION_SIZE)).astype(np.float32)
    with torch.no_grad():
        py_logits = actor(torch.from_numpy(sample)).numpy()
    ort_logits = sess.run(['logits'], {'input': sample})[0]
    max_diff = max(max_diff, float(np.max(np.abs(py_logits - ort_logits))))
print(f'max |PyTorch - ONNX| = {max_diff:.2e}')
assert max_diff < 1e-4, 'ONNX export diverged from PyTorch'


max |PyTorch - ONNX| = 1.82e-06


## 9. Download (Colab)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(CONFIG['model_out'])
else:
    print('local run — model saved to', CONFIG['model_out'])
